In [5]:
!pip install kafka-python requests faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.2/308.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.9 MB/s eta 0:00:0000:0100:01


In [3]:
from faker import Faker
fake = Faker()

In [8]:
print(fake.date_time_between(start_date='-30d', end_date='now'))
print(type(fake.date_time_between(start_date='-30d', end_date='now')))
x = fake.date_time_between(start_date='-30d', end_date='now')
print(fake.date_time_between(start_date=x, end_date='+10d'))

2025-04-25 19:33:47.925879
<class 'datetime.datetime'>
2025-05-14 14:19:23.707524


In [8]:
import json
from datetime import datetime
import time
from kafka import KafkaProducer
from faker import Faker
import random
import json
from datetime import datetime, timedelta

# Configuração do Kafka
KAFKA_BOOTSTRAP_SERVER = 'kafka:9092'
TOPICS = {
    "instagram": "instagram-post",
    "facebook": "facebook-post",
    "x": "x-post",
}

class FakeAPI:
    def __init__(self):
        self.fake = Faker()
        # Inicializar o produtor Kafka
        self.producer = KafkaProducer(
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVER,
            value_serializer=lambda v: json.dumps(v).encode('utf-8')
        )
    def generate_facebook_post(self):
        print("face")
        x = {
            "id": self.fake.uuid4(),
            "user_name": self.fake.name(),
            "post_content": self.fake.text(max_nb_chars=280),
            "created_at": self.fake.date_time_between(start_date='-30d', end_date='now').isoformat(),
            "likes": random.randint(0, 5000),
            "shared": random.randint(0, 5000),
            "comments": [
                {
                    "user": self.fake.name(),
                    "comment": self.fake.sentence(),
                    "timestamp": self.fake.date_time_between(start_date='-30d', end_date='now').isoformat()
                } for _ in range(random.randint(0, 10))
            ]
        }
        return x

    def generate_instagram_post(self):
        print("insta")
        x= {
            "id": self.fake.uuid4(),
            "user_handle": "@" + self.fake.user_name(),
            "caption": self.fake.text(max_nb_chars=150),
            "image_url": self.fake.image_url(),
            "posted_at": self.fake.date_time_between(start_date='-30d', end_date='now').isoformat(),
            "likes": random.randint(0, 10000),
            "hashtags": [f"#{self.fake.word()}" for _ in range(random.randint(1, 5))],
            "comments_count": random.randint(0, 100)
        }
        return x

    def generate_data(self, platform: str, count: int = 5):
        print(platform)
        if platform == "facebook":
            for _ in range(count):
                self.producer.send(TOPICS[platform], value=self.generate_facebook_post())
        elif platform == "instagram":
            for _ in range(count):
                self.producer.send(TOPICS[platform], value=self.generate_instagram_post())
        else:
            raise ValueError("Plataforma inválida. Use 'facebook' ou 'instagram'.")

    def run(self):
        self.generate_data("facebook", 30)
        self.generate_data("instagram", 3)

        # print("== = Facebook ===")
        # print(json.dumps(facebook_data, indent=2))
        #
        # print("\n=== Instagram ===")
        # print(json.dumps(instagram_data, indent=2))

if __name__ == "__main__":
    # print(f"Iniciando o processamento de mensagens do tópico {TOPIC_CONSUMER}...")
    fake_api = FakeAPI()
    fake_api.run()


facebook
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
face
instagram
insta
insta
insta


In [4]:
try:
    print(spark.read.parquet("hdfs://hadoop-namenode:8020/datalake/bronze/facebook/*.parquet").show(100,False))
except AnalysisException as e:
    print(f"Erro do Spark ao acessar os dados: {e}")

+----+---------+------------+----------+-----+------+--------+-----------------------+
|id  |user_name|post_content|created_at|likes|shares|comments|event_time             |
+----+---------+------------+----------+-----+------+--------+-----------------------+
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NULL |NULL  |NULL    |2025-05-11 20:45:31.329|
|NULL|NULL     |NULL        |NULL      |NUL

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, to_timestamp
from datetime import datetime, timedelta
spark = SparkSession.builder \
    .appName("SocialMediaSilver") \
    .getOrCreate()


current_time = datetime.utcnow()
window_start = current_time - timedelta(minutes=10)
window_end = current_time

# Função para carregar e transformar cada fonte
def process_social_data(path, source):
    # Define a janela de 10 minutos
    df = spark.read.parquet(path).filter(
        (col("event_time") >= window_start.isoformat()) &
        (col("event_time") < window_end.isoformat())
    )
    if source == "facebook":
        return df.select(
            col("user_name").alias("author"),
            col("post_content").alias("content"),
            to_timestamp("created_at").alias("post_date"),
            col("likes").alias("likes"),
            col("comments").alias("comments"),
            col("shares").alias("shares"),
        ).withColumn("source", lit("facebook"))

    elif source == "instagram":
        return df.select(
            col("user_handle").alias("author"),
            col("caption").alias("content"),
            to_timestamp("posted_at").alias("post_date"),
            col("likes").alias("likes"),
            col("metrics.comments").alias("comments"),
            lit(None).cast("int").alias("shares")
        ).withColumn("source", lit("instagram"))

    elif source == "twitter":
        return df.select(
            col("user.screen_name").alias("author"),
            col("tweet.text").alias("content"),
            to_timestamp("tweet.created_at").alias("post_date"),
            col("metrics.likes").alias("likes"),
            col("metrics.replies").alias("comments"),
            col("metrics.retweets").alias("shares")
        ).withColumn("source", lit("twitter"))



# Paths da Bronze
facebook_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/facebook", "facebook")
# instagram_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/instagram", "instagram")
# x_df = process_social_data("hdfs://hadoop-namenode:8020/datalake/bronze/x", "x")

# União e escrita na camada Silver
# silver_df = facebook_df.unionByName(instagram_df).unionByName(x_df)
facebook_df.show(100,False)

+------+-------+---------+-----+--------+------+--------+
|author|content|post_date|likes|comments|shares|source  |
+------+-------+---------+-----+--------+------+--------+
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL   |NULL     |NULL |NULL    |NULL  |facebook|
|NULL  |NULL  

In [ ]:

silver_df.coalese(1).write.mode("append").parquet("hdfs://hadoop-namenode:8020/datalake/silver/social_media/")
